# PDF → Chunking → Búsqueda semántica en Grafito

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpmanson/GrafitoDB/blob/main/examples/semantic/pdf_chunking_colab.ipynb)

Pipeline pedagógico sobre un PDF real de Anthropic:

**[Building Effective AI Agents: Architecture Patterns and Implementation Frameworks](https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf)**

Pasos que vamos a **ver** en el grafo:

1. Descargar el PDF y extraer texto  
2. Chunking jerárquico (`Document` / `Version` / `Section` / `Chunk`)  
3. Embeddings + búsqueda vectorial (consultas preparadas sobre el contenido)  
4. Expand + pack de contexto  
5. Hybrid RRF (si hay FTS5)  
6. Visualización PyVis en cada etapa  

> En Grafito: **1 nodo = 1 vector**. Un PDF largo se modela como muchos passages enlazados, no como multi-vector en un solo nodo.


## 0. Instalación (Colab)


In [ ]:
# Ejecutar una vez en Colab
%pip install -q "grafitodb[viz]>=0.4.0" pypdf sentence-transformers networkx requests
print("OK — paquetes instalados")


## 1. Helpers de visualización


In [ ]:
from __future__ import annotations

import re
import tempfile
from pathlib import Path
from typing import Any

import requests
from IPython.display import HTML, Markdown, display
from pypdf import PdfReader

from grafito import GrafitoDatabase
from grafito.document import DocumentIngestor, MarkdownChunker, TitleContextEnricher
from grafito.embedding_functions.base import EmbeddingFunction
from grafito.integrations import save_pyvis_html

DOC_COLORS = {
    "Document": "#264653",
    "DocumentVersion": "#2a9d8f",
    "Section": "#e9c46a",
    "Chunk": "#f4a261",
}


def node_display_label(node_id: Any, attrs: dict) -> str:
    props = attrs.get("properties") or {}
    labels = attrs.get("labels") or []
    if "Document" in labels:
        return f"PDF {(props.get('title') or props.get('document_key') or node_id)}"[:42]
    if "DocumentVersion" in labels:
        return f"v{props.get('generation', '?')} {props.get('status', '')}"
    if "Section" in labels:
        return f"S {(props.get('title') or '?')[:30]}"
    if "Chunk" in labels:
        text = (props.get("text") or "")[:34].replace("\n", " ")
        return f"#{props.get('global_seq', '?')} {text}..."
    return str(props.get("name") or props.get("title") or node_id)


def show_graph(
    db: GrafitoDatabase,
    *,
    title: str = "",
    include_ids: set[int] | None = None,
    highlight_ids: set[int] | None = None,
    height: str = "520px",
    physics: str = "spread",
) -> None:
    G = db.to_networkx()
    if include_ids is not None:
        G = G.subgraph([n for n in G.nodes if n in include_ids]).copy()

    highlight_ids = highlight_ids or set()
    for nid in list(G.nodes):
        attrs = G.nodes[nid]
        props = dict(attrs.get("properties") or {})
        labels = attrs.get("labels") or []
        if nid in highlight_ids:
            props["_viz_color"] = "#e63946"
        else:
            props["_viz_color"] = DOC_COLORS.get(labels[0] if labels else "", "#8ecae6")
        attrs["properties"] = props

    path = Path(tempfile.gettempdir()) / "grafito_step_graph.html"
    save_pyvis_html(
        G,
        path=str(path),
        notebook=False,
        directed=True,
        color_by_label=False,
        node_color_attr="_viz_color",
        label_fn=node_display_label,
        physics=physics,
        height=height,
        width="100%",
        bgcolor="#ffffff",
        font_color="#222222",
        cdn_resources="in_line",
    )
    raw = path.read_text(encoding="utf-8")
    if title:
        display(Markdown(f"### {title}"))
    display(HTML(f'<div style="border:1px solid #ddd;border-radius:8px;overflow:hidden">{raw}</div>'))


def legend() -> None:
    items = "".join(
        f'<span style="display:inline-block;margin:4px 12px 4px 0">'
        f'<span style="display:inline-block;width:12px;height:12px;background:{c};'
        f'border-radius:50%;margin-right:6px"></span>{name}</span>'
        for name, c in DOC_COLORS.items()
    )
    items += (
        '<span style="display:inline-block;margin:4px 12px 4px 0">'
        '<span style="display:inline-block;width:12px;height:12px;background:#e63946;'
        'border-radius:50%;margin-right:6px"></span>Hit / focus</span>'
    )
    display(HTML(f"<div style='font-family:sans-serif;font-size:14px'>{items}</div>"))


print("Helpers listos")
legend()


## 2. Descargar el PDF (fijo)

Fuente oficial Anthropic / Claude resources:

https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf


In [ ]:
PDF_URL = (
    "https://resources.anthropic.com/hubfs/"
    "Building%20Effective%20AI%20Agents-%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf"
)
PDF_PATH = Path("building_effective_ai_agents.pdf")

def download_pdf(url: str = PDF_URL, path: Path = PDF_PATH) -> Path:
    print(f"Descargando:\n  {url}")
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    path.write_bytes(resp.content)
    print(f"Guardado: {path.resolve()} ({path.stat().st_size:,} bytes)")
    return path

download_pdf()


## 3. Extraer texto y normalizar headings

`pypdf` devuelve texto plano. Promovemos títulos del whitepaper a markdown ATX (`#` / `##`) para el árbol de `Section`.


In [ ]:
# Titulos del PDF (ToC + secciones frecuentes). Orden: mas largos / especificos primero.
HEADING_SPECS: list[tuple[str, str]] = [
    ("#", "Building Effective AI Agents"),
    ("#", "Executive summary"),
    ("##", "The business case for AI agents"),
    ("##", "What organizations are achieving"),
    ("#", "Common use cases and applications for AI agents"),
    ("##", "Vertical spotlight: Financial services"),
    ("##", "Customer support and operations"),
    ("##", "Data analysis"),
    ("##", "Coding"),
    ("##", "Legal"),
    ("##", "Marketing"),
    ("#", "Common architecture patterns"),
    ("##", "Agent design best practices"),
    ("##", "When to use Skills"),
    ("##", "Example: Single-agent research agent"),
    ("##", "Single-agent systems"),
    ("##", "Multi-agent systems"),
    ("##", "Orchestrator"),
    ("##", "Parallelization"),
    ("##", "Routing"),
    ("##", "Evaluator"),
    ("#", "Looking forward: the future of building AI agents"),
    ("#", "Next steps"),
]


def pdf_to_text(path: Path) -> str:
    reader = PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages):
        t = page.extract_text() or ""
        pages.append(t)
        print(f"Pagina {i+1}/{len(reader.pages)}: {len(t):,} caracteres")
    return "\n\n".join(pages)


def promote_headings(plain: str, specs: list[tuple[str, str]] = HEADING_SPECS) -> str:
    lines = plain.splitlines()
    out: list[str] = []
    used: set[tuple[str, str]] = set()
    for line in lines:
        stripped = " ".join(line.split())
        matched = None
        for prefix, title in specs:
            if stripped == title or stripped.startswith(title):
                key = (prefix, title)
                if key not in used:
                    matched = f"{prefix} {title}"
                    used.add(key)
                    break
        out.append(matched if matched else line)
    return "\n".join(out)


RAW = pdf_to_text(PDF_PATH)
DOC_TEXT = promote_headings(RAW)

print("\n--- Vista previa (1200 chars) ---\n")
print(DOC_TEXT[:1200])
print(f"\n... total {len(DOC_TEXT):,} caracteres")
print("Headings ATX detectados:", sum(1 for ln in DOC_TEXT.splitlines() if ln.startswith("#")))


## 4. Ingest: chunking + embeddings


In [ ]:
from sentence_transformers import SentenceTransformer


class STEmbedder(EmbeddingFunction):
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self._dim = int(self.model.get_sentence_embedding_dimension())

    def __call__(self, input: list[str]) -> list[list[float]]:
        return self.model.encode(input, normalize_embeddings=True).tolist()

    @staticmethod
    def name() -> str:
        return "sentence_transformer_notebook"

    def default_space(self) -> str:
        return "cosine"

    def supported_spaces(self) -> list[str]:
        return ["cosine"]

    @staticmethod
    def build_from_config(config: dict) -> "STEmbedder":
        return STEmbedder(config.get("model_name", "sentence-transformers/all-MiniLM-L6-v2"))

    def get_config(self) -> dict:
        return {"model_name": self.model_name, "dim": self._dim}

    @staticmethod
    def validate_config(config: dict) -> None:
        return None

    @property
    def dimension(self) -> int:
        return self._dim


print("Cargando embeddings...")
embedder = STEmbedder()
print("dim =", embedder.dimension)

db = GrafitoDatabase(":memory:")
db.create_vector_index(
    "docs_chunks",
    dim=embedder.dimension,
    backend="bruteforce",
    embedding_function=embedder,
    options={"metric": "cosine"},
)

ing = DocumentIngestor(
    db,
    chunker=MarkdownChunker(max_chars=900, overlap=120),
    embed_index="docs_chunks",
    configure_fts=db.has_fts5(),
    enricher=TitleContextEnricher(),
    hierarchy="auto",
)

DOC_KEY = "anthropic/building-effective-ai-agents"
result = ing.ingest(
    DOC_TEXT,
    document_key=DOC_KEY,
    title="Building Effective AI Agents",
    source=PDF_URL,
    embed=True,
)
print(result)
print(
    f"hierarchy={result.hierarchy} sections={result.n_sections} "
    f"passages={result.n_passages} generation={result.generation}"
)


### 4.1 Grafo completo tras el ingest


In [ ]:
legend()
parent = db.match_nodes(labels=["Document"], properties={"document_key": DOC_KEY}, limit=1)[0]
ids = {parent.id}
for n in db.match_nodes(properties={"managed_by": "grafito.document"}):
    if n.properties.get("owner_document_id") == parent.id:
        ids.add(n.id)

show_graph(
    db,
    title="Grafo del whitepaper (Document -> Version -> Sections -> Chunks)",
    include_ids=ids,
    physics="spread",
    height="600px",
)

print("\nToC:")
for sec in ing.toc(DOC_KEY):
    print(f"  [{sec.node_key}] L{sec.level} {sec.title}")
    for c in sec.children[:10]:
        print(f"      [{c.node_key}] L{c.level} {c.title}")
    if len(sec.children) > 10:
        print(f"      ... +{len(sec.children)-10} more")


### 4.2 Solo arbol de secciones (ToC visual)


In [ ]:
sec_nodes = [
    n for n in db.match_nodes(labels=["Section"])
    if n.properties.get("owner_document_id") == parent.id
]
ver_nodes = [
    n for n in db.match_nodes(labels=["DocumentVersion"])
    if n.properties.get("owner_document_id") == parent.id
]
tree_ids = {parent.id} | {n.id for n in sec_nodes} | {n.id for n in ver_nodes}

show_graph(
    db,
    title="Solo Document + Version + Sections",
    include_ids=tree_ids,
    physics="spread",
    height="520px",
)


## 5. Consultas preparadas (contenido del PDF de Anthropic)

Cada query esta alineada a un tema del whitepaper (casos Coinbase/Tines/Gradient Labs, patrones single/multi-agent, Skills, routing, etc.).


In [ ]:
PREPARED_QUERIES: list[tuple[str, str]] = [
    (
        "How do AI agents differ from traditional automation scripts?",
        "Business case: agents choose tools and adapt vs rigid prewritten scripts.",
    ),
    (
        "Coinbase Claude customer support availability and scale",
        "Case study: 99.99% availability, thousands of messages/hour, 35-50 internal apps.",
    ),
    (
        "Tines security workflow orchestration time to value improvement",
        "Case study: multi-step security ops collapsed into agent workflows (~100x).",
    ),
    (
        "Gradient Labs financial services customer support resolution rates",
        "Case study: 80-90% resolution with limited human intervention.",
    ),
    (
        "retail bank credit risk memo productivity gains percentage",
        "Results: 20-60% productivity gains and faster credit turnaround.",
    ),
    (
        "When should I use a single-agent system versus multi-agent orchestration?",
        "Architecture patterns: match problem shape to single vs multi-agent.",
    ),
    (
        "What are agent Skills and when should organizations use them?",
        "Skills for standardized workflows, tool integrations, compliance practices.",
    ),
    (
        "routing pattern to specialized agents",
        "Architecture: route tasks to the right specialized agent.",
    ),
    (
        "parallelization of independent agent work streams",
        "Architecture: run independent agent work in parallel.",
    ),
    (
        "evaluator optimizer feedback loop for agent quality",
        "Architecture: evaluate outputs and iterate on agent behavior.",
    ),
    (
        "customer support operations use cases for AI agents",
        "Use cases: escalations, languages, complex Tier 2+ issues.",
    ),
    (
        "security and compliance frameworks for autonomous agent systems",
        "Governance: protect sensitive data with autonomous systems.",
    ),
    (
        "LangGraph or Mastra frameworks for multi-step agent reasoning",
        "Implementation frameworks mentioned for agent workflows.",
    ),
    (
        "coding agents that write document and maintain code",
        "Use case chapter: coding productivity.",
    ),
]


def run_query(query: str, note: str = "", k: int = 3, show: bool = True):
    print("=" * 72)
    print(f"QUERY: {query}")
    if note:
        print(f"NOTE:  {note}")
    hits = ing.search(query, k=k)
    for i, h in enumerate(hits, 1):
        text = (h.node.properties.get("text") or "").replace("\n", " ")
        print(f"\n  {i}. score={h.score:.3f} seq={h.global_seq}")
        print(f"     {text[:220]}...")
    if show and hits:
        hit_ids = {h.node.id for h in hits}
        extra = set()
        ex0 = ing.expand(hits[0].node, window=0, include_ancestors=True)
        if ex0.section:
            extra.add(ex0.section.id)
        extra.update(a.id for a in ex0.ancestors)
        extra.add(parent.id)
        show_graph(
            db,
            title=f"Hits: {query[:64]}...",
            include_ids=ids,
            highlight_ids=hit_ids | extra,
            physics="spread",
            height="480px",
        )
    return hits


# Tres demos representativas (casos + patron de arquitectura)
for q, note in PREPARED_QUERIES[:3]:
    run_query(q, note, k=3, show=True)


### 5.1 Proba otra query del catalogo


In [ ]:
for i, (q, note) in enumerate(PREPARED_QUERIES):
    print(f"[{i:2d}] {q}")

idx = 5  # cambia 0 .. len(PREPARED_QUERIES)-1
q, note = PREPARED_QUERIES[idx]
_ = run_query(q, note, k=4, show=True)


## 6. Expand + pack (contexto para un LLM)


In [ ]:
focus_hits = run_query(
    "Coinbase Claude customer support availability and scale",
    "Pack del mejor passage + vecinos de lectura",
    k=3,
    show=False,
)
top = focus_hits[0]
expanded = ing.expand(top.node, window=1, include_parent=True, include_ancestors=True)
packed = ing.pack(expanded, max_chars=2200, include_citations=True)

print("Section:", None if not expanded.section else expanded.section.properties.get("title"))
print("Ancestors:", [a.properties.get("title") for a in expanded.ancestors])
print("Passages window global_seq:", [p.properties.get("global_seq") for p in expanded.passages])
print("truncated:", packed.truncated)
print("\n----- PACKED CONTEXT -----\n")
print(packed.text[:2500] + ("..." if len(packed.text) > 2500 else ""))

window_ids = {p.id for p in expanded.passages}
if expanded.section:
    window_ids.add(expanded.section.id)
window_ids.update(a.id for a in expanded.ancestors)
window_ids.add(parent.id)

show_graph(
    db,
    title="Ventana expand (passages vecinos + seccion)",
    include_ids=window_ids,
    highlight_ids={top.node.id},
    physics="compact",
    height="480px",
)


## 7. Hybrid search (vector + FTS + RRF)


In [ ]:
if db.has_fts5():
    for q in [
        "Coinbase 99.99% availability",
        "100x time-to-value Tines",
        "LangGraph Mastra",
        "80-90% resolution Gradient Labs",
    ]:
        print("\n" + "=" * 72)
        print("HYBRID:", q)
        for i, h in enumerate(ing.hybrid_search(q, k=3), 1):
            text = (h.node.properties.get("text") or "").replace("\n", " ")[:170]
            print(f"  {i}. rrf={h.score:.4f}  {text}...")
else:
    print("FTS5 no disponible — se omite hybrid_search.")


## 8. Ejercicios

1. Cambia `idx` en el catalogo y contrasta **vector** vs **hybrid** en nombres propios (Coinbase, Tines, Gradient Labs).
2. Compara `window=0` vs `window=2` en expand sobre la misma query.
3. Fuerza `hierarchy=False` en un nuevo `DocumentIngestor` y mira el grafo (solo Chunks planos).
4. (Avanzado) Usa `tree_select` con un LLM que elija `node_key` del ToC para multi-agent orchestration.

### Referencias

- PDF: [Building Effective AI Agents (Anthropic)](https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf)
- Docs: [Document Chunking](https://jpmanson.github.io/GrafitoDB/search/document-chunking/)
- Visualizacion: [Visualization](https://jpmanson.github.io/GrafitoDB/integrations/visualization/)


In [ ]:
db.close()
print("Sesion cerrada.")
